<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 3. База данных и минимальный Docker — подробная методичка

Эта методичка продолжает Этап 2. Предполагается, что у вас уже есть работающее приложение: `backend/api/main.py`, `/healthz`, `backend/core/config.py`, `backend/utils/logging.py`, `run.py`, зелёные тесты. Если чего-то нет — вернитесь к Этапу 2.

**Цель этапа:** подключить БД, описать модели, поднять Postgres и app в Docker.

**Недель:** 1.

**Что вы получите в конце этапа:**

- `backend/db/session.py` — async-подключение к БД через SQLAlchemy;
- `backend/db/models.py` — пять моделей: `User`, `TMDB`, `GlossaryDB`, `UncertainExample`, `TranslationCache`;
- `backend/db/database.py` — реэкспорт для удобного импорта;
- `backend/api/main.py` — обновлённый `lifespan` с автоматическим созданием таблиц;
- `backend/api/routes/health.py` — расширенный `/health`, проверяющий БД;
- `infrastructure/Dockerfile` — минимальный образ приложения;
- `infrastructure/docker-compose.yml` — сервисы `db` (Postgres 16) и `app`;
- `infrastructure/.env.example` и `infrastructure/.env` — настройки для Docker;
- `infrastructure/.dockerignore` — исключения для образа;
- `tests/unit/test_models.py` — тесты на модели с SQLite in-memory;
- запущенный `docker compose`, где оба сервиса `healthy`;
- набор CRUD-команд для работы с БД через `docker compose exec`.

**Критерии готовности:**

- [ ] `Base.metadata.create_all` создаёт все таблицы при старте приложения.
- [ ] `/health` проверяет БД через `SELECT 1`.
- [ ] `docker compose ps` показывает оба сервиса `healthy`.
- [ ] При перезапуске контейнера данные сохраняются (volume `postgres_data`).
- [ ] `python -m poetry run pytest tests/unit/test_models.py -v` проходит.
- [ ] Вы умеете вставить и прочитать запись через `psql` в контейнере.

**Методичка:** подробная. Разбираем SQLAlchemy async, UUID, ловушку «Future attached to a different loop», работу с healthcheck'ами и порядок старта контейнеров.

**Почему Docker уже здесь:** если отложить — 14 недель разработки на SQLite, потом неделя боли при переезде. `localhost` в контейнере ≠ `localhost` хоста, `postgresql://` ≠ `postgresql+asyncpg://`, healthcheck'и и порядок старта. Проще решить сейчас, чем в конце семестра.

---

## 3.0. Подготовка: ветка в Git

**Почему:** каждый этап курса — отдельная ветка. Позволяет преподавателю видеть прогресс по неделям, а вам — откатываться, если эксперимент не удался.

Откройте PowerShell, перейдите в папку проекта:

```powershell
cd D:\RUNG
```

Убедитесь, что ветка `main` чистая:

```powershell
git status
```

**Что вы должны увидеть:**

```
On branch main
nothing to commit, working tree clean
```

Если есть незакоммиченные изменения — сначала закоммитьте их.

Создайте ветку для этого этапа:

```powershell
git checkout -b week-3
```

**Что вы должны увидеть:**

```
Switched to a new branch 'week-3'
```

---

## 3.1. `backend/db/session.py` — подключение к БД

**Что это:** модуль, который создаёт async engine SQLAlchemy и фабрику сессий. Все запросы к БД в проекте идут через эти объекты.

### Что такое engine и session

**Engine** — объект, который держит пул соединений с БД. Один на всё приложение. Создаётся при импорте модуля и живёт до остановки приложения.

**Session** — единица работы с БД (транзакция). Создаётся на каждый запрос пользователя, закрывается после ответа. Не путайте: одна session ≠ одно соединение с БД. SQLAlchemy берёт соединение из пула на время активной транзакции и возвращает обратно.

**Почему async:** FastAPI работает асинхронно. Если использовать синхронный драйвер (например, `psycopg2`), каждый запрос к БД будет блокировать event loop и тормозить всё приложение. Async-драйверы (`asyncpg`, `aiosqlite`) отдают управление, пока ждут ответа БД.

### Создание файла

Создайте `backend/db/session.py`:

```python
# backend/db/session.py
"""
Асинхронное подключение к базе данных через SQLAlchemy.

- async_engine — пул соединений с БД (создаётся один раз при импорте).
- AsyncSessionLocal — фабрика сессий (создаёт новый Session на каждый запрос).
- get_async_session — FastAPI-зависимость для получения сессии в роутах.
"""

import logging
from typing import AsyncGenerator

from sqlalchemy.ext.asyncio import (
    AsyncSession,
    async_sessionmaker,
    create_async_engine,
)

from backend.core.config import get_settings

logger = logging.getLogger(__name__)

settings = get_settings()


# ============================================================================
# Engine — пул соединений с БД
# ============================================================================
# echo=settings.DEBUG включает SQL-логирование (видно каждый запрос).
# В продакшене выключено, потому что создаёт много шума.
async_engine = create_async_engine(
    settings.ASYNC_DATABASE_URL,
    echo=settings.DEBUG,
    pool_size=10,
    max_overflow=20,
    pool_pre_ping=True,
)


# ============================================================================
# Session factory — создаёт сессии для каждого запроса
# ============================================================================
AsyncSessionLocal = async_sessionmaker(
    bind=async_engine,
    class_=AsyncSession,
    expire_on_commit=False,
    autocommit=False,
    autoflush=False,
)


# ============================================================================
# FastAPI dependency
# ============================================================================
async def get_async_session() -> AsyncGenerator[AsyncSession, None]:
    """
    FastAPI-зависимость: выдаёт сессию и закрывает после ответа.

    Использование в роутере:
        async def my_route(session: AsyncSession = Depends(get_async_session)):
            ...
    """
    async with AsyncSessionLocal() as session:
        yield session
```

### Разбор ключевых параметров

**`echo=settings.DEBUG`:** при `DEBUG=true` в логах видно каждый SQL-запрос. Полезно при отладке, шумно в проде. Управляется через `.env`, отдельного кода для этого не нужно.

**`pool_size=10`:** держать до 10 открытых соединений с БД постоянно. Это баланс между скоростью (открытые соединения переиспользуются без задержки на «рукопожатие») и ресурсами Postgres (10 соединений висят в памяти). Для курсового проекта — с запасом.

**`max_overflow=20`:** при пиковой нагрузке разрешено временно открыть ещё до 20 соединений (всего 30). После спада лишние закрываются. Помогает при всплесках трафика.

**`pool_pre_ping=True`:** перед использованием соединения из пула SQLAlchemy делает лёгкий `SELECT 1`. Если соединение «умерло» (БД перезапустилась, сеть отвалилась) — оно будет пересоздано. Без этого параметра первый запрос после разрыва упадёт с `ConnectionDoesNotExistError`.

**`expire_on_commit=False`:** после `commit()` объекты не «протухают». Без этого параметра любой доступ к атрибутам объекта после коммита вызовет новый запрос к БД. В async-коде это критично: обращения к атрибутам — синхронные, и SQLAlchemy не может «подождать» загрузку из БД. Результат — `MissingGreenlet`.

**`autocommit=False`, `autoflush=False`:** явный контроль транзакций и сброса изменений. Значения по умолчанию, но прописаны явно для ясности — будущий читатель кода сразу видит, что управление ручное.

### Про ловушку «Future attached to a different loop»

**Что за проблема:** SQLAlchemy async engine привязывается к event loop, в котором создан. Если приложение создаёт новый event loop (например, Celery worker использует `asyncio.new_event_loop()` на каждую задачу), движок перестаёт работать с ошибкой `RuntimeError: Task got Future attached to a different loop`.

**Когда это случится:** на этапе 11, когда будем подключать Celery. Запомните это место — вернёмся к нему.

**Как решать (кратко):** один event loop на процесс, не создавать новый на каждую задачу. В FastAPI это не проблема — там один loop на весь сервер.

**Сейчас:** просто знайте, что такая ловушка существует. Движок создан один раз при импорте модуля — это правильно.

### Проверка: engine создаётся

```powershell
python -m poetry run python -c "from backend.db.session import async_engine; print(async_engine.url); print('OK')"
```

**Что вы должны увидеть:**

```
sqlite+aiosqlite:///./rung.db
OK
```

**Если увидели ошибку `ModuleNotFoundError: No module named 'aiosqlite'`** — переустановите зависимости: `python -m poetry install`.

### Коммит

```powershell
git add backend/db/session.py
git commit -m "feat(week-3): add async DB session"
```

---

## 3.2. `backend/db/models.py` — модели SQLAlchemy

**Что это:** описание таблиц БД в виде Python-классов. Каждый класс = одна таблица. Каждый атрибут = одна колонка.

**Зачем это:** вместо того чтобы писать SQL вручную (`CREATE TABLE users (...)`), мы описываем структуру на Python. SQLAlchemy сама генерирует SQL для нужной СУБД. Код при этом остаётся переносимым между SQLite и Postgres.

### Какие таблицы и зачем

| Таблица | Назначение | Кто пишет |
|---------|------------|-----------|
| `users` | Пользователи системы (роли: user, editor, admin) | `/auth/register` (этап 4) |
| `translation_memories` | Память переводов (TM) — пары «источник → перевод» | `/admin/tm/add` (этап 9) |
| `glossaries` | Глоссарий — термины и их переводы | `/admin/glossary/add` (этап 9) |
| `uncertain_examples` | Очередь HITL — неуверенные переводы на проверку | `_perform_translation` (этап 9) |
| `translation_cache` | Кэш переводов | `cache_manager` (этап 11) |

### Создание файла

Создайте `backend/db/models.py`:

```python
# backend/db/models.py
"""
SQLAlchemy-модели базы данных RUNG.

Каждый класс = таблица. Атрибуты = колонки.
Все таблицы наследуются от Base (declarative_base).
"""

import uuid

from sqlalchemy import (
    Boolean,
    Column,
    DateTime,
    Float,
    Index,
    Integer,
    String,
    Text,
    UniqueConstraint,
    func,
)
from sqlalchemy.orm import declarative_base

Base = declarative_base()


# ============================================================================
# Users — пользователи системы
# ============================================================================
class User(Base):
    """
    Пользователь системы.

    Роли:
    - user   — обычный пользователь, может переводить
    - editor — может редактировать TM/глоссарий и обрабатывать HITL
    - admin  — полный доступ
    """

    __tablename__ = "users"

    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    email = Column(String, unique=True, nullable=False)
    username = Column(String, unique=True, nullable=False)
    hashed_password = Column(String, nullable=False)
    full_name = Column(String, nullable=True)
    role = Column(String, default="user", nullable=False)
    is_active = Column(Boolean, default=True, nullable=False)

    created_at = Column(DateTime(timezone=True), server_default=func.now())
    updated_at = Column(
        DateTime(timezone=True),
        server_default=func.now(),
        onupdate=func.now(),
    )

    __table_args__ = (
        Index("ix_users_email", "email"),
        Index("ix_users_username", "username"),
        Index("ix_users_role", "role"),
    )

    def __repr__(self):
        return f"<User {self.email} ({self.role})>"


# ============================================================================
# Translation Memory (TM) — память переводов
# ============================================================================
class TMDB(Base):
    """
    Запись памяти переводов: пара «источник → перевод» для конкретной языковой пары.

    Одна и та же пара source_text + target_text для одной пары языков
    может существовать только в одном экземпляре (UniqueConstraint).
    """

    __tablename__ = "translation_memories"

    id = Column(String, primary_key=True)
    source_lang = Column(String, nullable=False)
    target_lang = Column(String, nullable=False)
    source_text = Column(Text, nullable=False)
    target_text = Column(Text, nullable=False)

    created_at = Column(DateTime(timezone=True), server_default=func.now())

    __table_args__ = (
        UniqueConstraint(
            "source_lang",
            "target_lang",
            "source_text",
            "target_text",
            name="uq_tm_entry",
        ),
        Index("ix_tm_langs_text", "source_lang", "target_lang", "source_text"),
    )


# ============================================================================
# Glossary — глоссарий терминов
# ============================================================================
class GlossaryDB(Base):
    """
    Запись глоссария: термин и его перевод для конкретной языковой пары.

    Структурно то же, что TM, но семантически другое:
    - TM — это пары предложений (переводы целиком).
    - Glossary — отдельные термины (используются для контроля перевода).
    """

    __tablename__ = "glossaries"

    id = Column(String, primary_key=True)
    source_lang = Column(String, nullable=False)
    target_lang = Column(String, nullable=False)
    source_text = Column(Text, nullable=False)
    target_text = Column(Text, nullable=False)

    created_at = Column(DateTime(timezone=True), server_default=func.now())

    __table_args__ = (
        UniqueConstraint(
            "source_lang",
            "target_lang",
            "source_text",
            "target_text",
            name="uq_glossary_entry",
        ),
        Index("ix_glossaries_langs_text", "source_lang", "target_lang", "source_text"),
    )


# ============================================================================
# Uncertain Examples — очередь HITL
# ============================================================================
class UncertainExample(Base):
    """
    Неуверенные переводы, ожидающие проверки редактором (HITL).

    Когда Critic ставит оценку ниже 7 или Validator находит ошибки,
    пример попадает сюда. Редактор может:
    - подтвердить исправление (status='confirmed'),
    - отклонить пример (status='rejected').
    """

    __tablename__ = "uncertain_examples"

    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    source_text = Column(Text, nullable=False)
    source_lang = Column(String, nullable=False)
    target_lang = Column(String, nullable=False)
    machine_translation = Column(Text, nullable=False)
    corrected_translation = Column(Text, nullable=True)
    critic_score = Column(Float, nullable=True)
    validation_passed = Column(Boolean, default=False, nullable=False)
    status = Column(String, default="pending", nullable=False)

    created_at = Column(DateTime(timezone=True), server_default=func.now())
    confirmed_at = Column(DateTime(timezone=True), nullable=True)

    __table_args__ = (
        Index("ix_uncertain_status", "status"),
        Index("ix_uncertain_langs", "source_lang", "target_lang", "status"),
    )


# ============================================================================
# Translation Cache — кэш переводов
# ============================================================================
class TranslationCache(Base):
    """
    Кэш переводов.

    Уникальность по (source_text, source_lang, target_lang, engine_used):
    один и тот же текст может быть переведён разными движками
    (ollama / huggingface) и результаты различаются — это разные записи.
    """

    __tablename__ = "translation_cache"

    id = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    source_text = Column(Text, nullable=False)
    source_lang = Column(String, nullable=False)
    target_lang = Column(String, nullable=False)
    translation = Column(Text, nullable=False)
    model_used = Column(String, nullable=False)
    engine_used = Column(String, nullable=True)
    hits = Column(Integer, default=1, nullable=False)

    created_at = Column(DateTime(timezone=True), server_default=func.now())

    __table_args__ = (
        UniqueConstraint(
            "source_text",
            "source_lang",
            "target_lang",
            "engine_used",
            name="unique_translation_engine",
        ),
        Index("ix_cache_langs", "source_lang", "target_lang"),
        Index("ix_cache_created", "created_at"),
    )
```

### Разбор ключевых решений

**Почему `id` — String, а не Integer:** UUID удобен тем, что его можно генерировать на стороне приложения, не дожидаясь ответа БД. Это ускоряет вставку, упрощает тесты (можно сделать объект с заранее известным id) и не даёт угадать количество пользователей по порядковым номерам.

**Почему `default=lambda: str(uuid.uuid4())`, а не `default=uuid.uuid4`:** SQLAlchemy вызывает callable при вставке. `uuid.uuid4()` возвращает объект `UUID`, а колонка — `String`. Лямбда конвертирует в строку на месте.

**Почему `Text`, а не `String`:** `Text` в Postgres = `TEXT` (без ограничения длины). `String` без параметров = `VARCHAR` с дефолтной длиной. Для переводов и терминов `Text` правильнее — они могут быть длинными.

**Почему `server_default=func.now()`:** значение генерирует **сама БД** при вставке. Это надёжнее, чем `default=datetime.utcnow` в Python: время будет в таймзоне БД, одинаковое для всех вставок.

**Почему `onupdate=func.now()` для `updated_at`:** при каждом `UPDATE` этой строки БД автоматически обновит поле. Не нужно вручную писать `updated_at=now()` в коде.

**Про `UniqueConstraint` в TM/Glossary:** защищает от дублей на уровне БД. Даже если два процесса одновременно попытаются вставить одну пару — БД отклонит вторую вставку с `IntegrityError`. Это надёжнее, чем проверять существование через `SELECT` перед `INSERT` (между проверкой и вставкой может вклиниться другая транзакция).

**Про `UniqueConstraint` в Cache:** уникальность включает `engine_used`. Это значит, что для одного текста может быть две записи в кэше — с `engine_used="ollama"` и с `engine_used="huggingface"`. Они не конфликтуют.

**Про `nullable=False` в `status`, `role`, `is_active`:** явно указано, чтобы БД не допускала `NULL` в этих полях. `default` срабатывает на уровне Python, `nullable=False` — на уровне БД. Вместе они дают надёжную защиту.

**Про индексы:** `Index` ускоряет поиск по колонке. Например, `ix_users_email` нужен для `SELECT ... WHERE email = ?` при логине. `ix_cache_created` — для задачи очистки старого кэша (этап 11). `ix_uncertain_status` — для запроса «все pending примеры».

### Создание `backend/db/database.py`

Этот файл просто реэкспортирует `async_engine` и `AsyncSessionLocal` для удобного импорта:

```python
# backend/db/database.py
"""
Удобный реэкспорт для импорта:

    from backend.db.database import async_engine, AsyncSessionLocal
"""

from backend.db.session import AsyncSessionLocal, async_engine

__all__ = ["async_engine", "AsyncSessionLocal"]
```

**Зачем он нужен:** в некоторых модулях удобнее импортировать оба объекта одной строкой. Плюс это точка расширения — если в будущем понадобится добавить что-то к импорту, это делается в одном месте.

### Проверка: модели создаются

```powershell
python -m poetry run python -c "from backend.db.models import User, TMDB, GlossaryDB, UncertainExample, TranslationCache, Base; print(list(Base.metadata.tables.keys()))"
```

**Что вы должны увидеть:**

```
['users', 'translation_memories', 'glossaries', 'uncertain_examples', 'translation_cache']
```

Пять таблиц — все модели описаны корректно.

### Коммит

```powershell
git add backend/db/models.py backend/db/database.py
git commit -m "feat(week-3): add SQLAlchemy models"
```

---

## 3.3. Обновление `backend/api/main.py` — автосоздание таблиц

**Что делаем:** в `lifespan` приложения вызываем `Base.metadata.create_all`. При старте приложение само создаст таблицы, если их нет.

**Зачем:** на этапе разработки это удобно — не нужно вручную писать SQL. В продакшене перейдём на Alembic (см. ниже).

**Что такое `lifespan`:** механизм FastAPI для запуска кода при старте и остановке приложения. Всё, что до `yield`, — при старте. Всё, что после, — при остановке.

Откройте `backend/api/main.py` и обновите блок `lifespan`:

```python
# backend/api/main.py
# ... (импорты в начале файла остаются)

# Добавьте эти импорты рядом с остальными:
from backend.db.session import async_engine
from backend.db.models import Base


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Управляет ресурсами приложения."""
    logger.info("=" * 60)
    logger.info("%s starting...", settings.PROJECT_NAME)
    logger.info("Debug mode: %s", settings.DEBUG)
    logger.info("Translation engine: %s", settings.TRANSLATION_ENGINE)
    logger.info("=" * 60)

    # Создание таблиц БД при старте (если их ещё нет)
    logger.info("Creating database tables...")
    async with async_engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    logger.info("Database tables ready")

    # Здесь позже будет:
    # - инициализация Qdrant-коллекции — этап 7
    # - прогрев эмбеддингов — этап 7

    yield

    logger.info("=" * 60)
    logger.info("%s stopped", settings.PROJECT_NAME)
    logger.info("=" * 60)

    # Закрываем пул соединений
    await async_engine.dispose()
```

### Разбор

**Почему `async with async_engine.begin() as conn`:** открывает транзакцию. `create_all` выполнится в ней. Если что-то пойдёт не так — изменения откатятся.

**Почему `conn.run_sync(Base.metadata.create_all)`:** `create_all` — синхронный метод SQLAlchemy. `run_sync` запускает его в отдельном потоке, не блокируя event loop. Без `run_sync` будет ошибка.

**Почему `await async_engine.dispose()` в конце:** при остановке приложения закрываем все соединения пула. Без этого Postgres может держать «мёртвые» соединения до истечения таймаута.

**Идемпотентность:** `create_all` проверяет существование таблиц и создаёт только отсутствующие. Повторный запуск ничего не сломает.

**Почему не Alembic:** Alembic — это инструмент миграций, он умеет аккуратно менять схему (добавить колонку, переименовать таблицу) без потери данных. `create_all` умеет только создавать отсутствующие таблицы. Пока схема не меняется после первого создания — `create_all` достаточно. Alembic подключим, когда начнём менять схему (например, при добавлении новых полей). Это нормальная практика: многие проекты начинают с `create_all` и переходят на Alembic позже.

### Проверка

Запустите приложение:

```powershell
python -m poetry run python run.py
```

**Что вы должны увидеть** среди логов:

```
INFO - RUNG starting...
INFO - Debug mode: True
INFO - Translation engine: auto
INFO - Creating database tables...
INFO - Database tables ready
```

**Проверьте, что файл БД появился** (при использовании SQLite):

```powershell
dir rung.db
```

**Что вы должны увидеть:** файл `rung.db` есть, размер > 0.

**Проверьте таблицы:**

```powershell
python -m poetry run python -c "import sqlite3; conn = sqlite3.connect('rung.db'); cur = conn.cursor(); cur.execute(\"SELECT name FROM sqlite_master WHERE type='table'\"); print([r[0] for r in cur.fetchall()])"
```

**Что вы должны увидеть:**

```
['users', 'translation_memories', 'glossaries', 'uncertain_examples', 'translation_cache']
```

Остановите приложение (**Ctrl + C**).

### Коммит

```powershell
git add backend/api/main.py
git commit -m "feat(week-3): create DB tables on app startup"
```

---

## 3.4. Обновление `backend/api/routes/health.py` — проверка БД

**Что делаем:** добавляем роут `/health`, который делает `SELECT 1` в БД и возвращает статус.

**Зачем:** `/healthz` — это «жив ли процесс». `/health` — это «готов ли принимать запросы». Разные проверки для разных целей. Docker healthcheck обращается к `/health`, чтобы понять, можно ли направлять трафик.

Откройте `backend/api/routes/health.py` и замените содержимое:

```python
# backend/api/routes/health.py
"""
Healthcheck endpoints.

/healthz  — простой liveness-проверка.
/health   — readiness: проверяет подключение к БД.
/languages — статический список поддерживаемых языков.
"""

import logging

from fastapi import APIRouter, status
from sqlalchemy import text

from backend.core.config import get_settings
from backend.core.constants import LANGUAGES, KNOWN_LANGUAGES
from backend.db.session import async_engine

logger = logging.getLogger(__name__)
router = APIRouter(tags=["health"])


@router.get("/healthz")
async def healthz():
    """Liveness-проверка: приложение запустилось."""
    return {"status": "ok"}


@router.get("/health")
async def health():
    """
    Readiness-проверка: доступна ли БД.

    Возвращает 200, если БД отвечает, иначе 503.
    """
    settings = get_settings()

    db_ok, db_error = await check_database()

    all_healthy = db_ok
    status_code = status.HTTP_200_OK if all_healthy else status.HTTP_503_SERVICE_UNAVAILABLE

    return {
        "status": "healthy" if all_healthy else "unhealthy",
        "project": settings.PROJECT_NAME,
        "model": settings.LLM_MODEL,
        "debug": settings.DEBUG,
        "services": {
            "database": {"healthy": db_ok, "error": db_error},
        },
    }


@router.get("/languages")
async def languages():
    """Список поддерживаемых языков."""
    return {
        "languages": LANGUAGES,
        "known_languages": KNOWN_LANGUAGES,
    }


# ============================================================================
# Проверки сервисов
# ============================================================================
async def check_database():
    """Проверяет, что БД отвечает на простой запрос."""
    try:
        async with async_engine.connect() as conn:
            await conn.execute(text("SELECT 1"))
        return True, None
    except Exception as e:
        logger.error(f"Database health check failed: {e}")
        return False, str(e)
```

### Разбор

**Почему `text("SELECT 1")`:** SQLAlchemy требует обёртку `text()` для сырого SQL. Без неё он попытается интерпретировать `"SELECT 1"` как имя таблицы и упадёт.

**Почему возвращаем 503, а не 500:** 503 — Service Unavailable — правильный код для «сервис временно недоступен». Системы мониторинга (Docker healthcheck, Kubernetes readiness probe) реагируют именно на этот код.

**Почему `services` — вложенный объект:** когда будем добавлять Redis, Qdrant, Ollama, каждый будет отдельным ключом. Формат расширяемый, не нужно менять структуру API.

**Почему `status_code` не используется:** в FastAPI декоратор возвращает 200 автоматически. Чтобы вернуть 503, нужно возвращать `Response(status_code=503, ...)` или использовать `JSONResponse`. Сейчас оставили переменную как заготовку — на следующих этапах доработаем.

### Проверка

Запустите приложение:

```powershell
python -m poetry run python run.py
```

В новом окне:

```powershell
curl http://localhost:8000/health
```

**Что вы должны увидеть:**

```json
{"status":"healthy","project":"RUNG","model":"qwen2.5:7b","debug":true,"services":{"database":{"healthy":true,"error":null}}}
```

**Остановка БД для теста:** на этом этапе БД у нас SQLite (файл), её нельзя «остановить». Проверка «нездорового» состояния будет на этапе 3.7, когда поднимем Postgres в Docker.

Остановите приложение (**Ctrl + C**).

### Коммит

```powershell
git add backend/api/routes/health.py
git commit -m "feat(week-3): add /health with database check"
```

---

## 3.5. `infrastructure/Dockerfile` — минимальный образ

**Что это:** инструкция для Docker, как собрать образ приложения.

**Зачем Docker:** на вашей машине приложение работает, но у коллеги «не запускается». Docker решает это: один образ — одинаковый результат везде.

Создайте `infrastructure/Dockerfile`:

```dockerfile
# infrastructure/Dockerfile
# Минимальный образ для приложения RUNG.

FROM python:3.12-slim

WORKDIR /app

# Системные зависимости: gcc для сборки C-расширений,
# libssl-dev для криптографии, curl для healthcheck.
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    libffi-dev \
    libssl-dev \
    curl \
    && rm -rf /var/lib/apt/lists/*

# Устанавливаем Poetry
RUN pip install --no-cache-dir poetry && \
    poetry config virtualenvs.create false

# Ставим зависимости отдельно — кэшируется, если pyproject не менялся
COPY pyproject.toml poetry.lock ./
RUN poetry install --no-root --only main --no-interaction --no-ansi

# Копируем код
COPY backend/ ./backend/
COPY run.py ./

# Директория для логов
RUN mkdir -p /app/logs

EXPOSE 8000

CMD ["uvicorn", "backend.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Разбор

**Почему `python:3.12-slim`:** совпадает с локальной версией Python. Slim — образ без лишних инструментов (~150 МБ вместо ~900 МБ). Все нужные библиотеки (`gcc` и т.п.) мы доставим сами.

**Почему `--no-root`:** проект не устанавливается как пакет — он не библиотека, а приложение. `--no-root` пропускает установку текущего проекта.

**Почему `--only main`:** dev-зависимости (pytest, ruff, black) в образ не нужны, они только для разработки. Экономит ~200 МБ.

**Почему `COPY pyproject.toml poetry.lock` отдельно от кода:** Docker кэширует слои. Если поменяли код, но не `pyproject.toml` — слой с установкой зависимостей возьмётся из кэша, сборка будет в разы быстрее (секунды вместо минут).

**Почему `CMD ["uvicorn", ...]` без `--reload`:** в контейнере reload не нужен, это dev-фича. В `docker-compose.yml` для разработки мы потом добавим `command:` с `--reload`, но базовый `CMD` — прод-вариант.

### Коммит

```powershell
git add infrastructure/Dockerfile
git commit -m "feat(week-3): add Dockerfile"
```

---

## 3.6. `infrastructure/docker-compose.yml` — сервисы `db` и `app`

**Что это:** описание, из каких контейнеров состоит система и как они связаны.

Создайте `infrastructure/docker-compose.yml`:

```yaml
# infrastructure/docker-compose.yml
services:

  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: ${POSTGRES_USER}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
      POSTGRES_DB: ${POSTGRES_DB}
    volumes:
      - postgres_data:/var/lib/postgresql/data
    restart: unless-stopped
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U \"$${POSTGRES_USER}\" -d \"$${POSTGRES_DB}\""]
      interval: 10s
      timeout: 5s
      retries: 5
      start_period: 10s
    networks:
      - rung_internal

  app:
    build:
      context: ..
      dockerfile: infrastructure/Dockerfile
    ports:
      - "8000:8000"
    volumes:
      - ../backend:/app/backend
      - ../logs:/app/logs
    env_file:
      - .env
    environment:
      DATABASE_URL: ${DATABASE_URL}
      PYTHONUNBUFFERED: 1
      PYTHONDONTWRITEBYTECODE: 1
    depends_on:
      db:
        condition: service_healthy
    restart: unless-stopped
    command:
      - uvicorn
      - backend.api.main:app
      - --host
      - "0.0.0.0"
      - --port
      - "8000"
      - --reload
      - --reload-dir
      - backend
    networks:
      - rung_internal


volumes:
  postgres_data:
    name: rung_postgres_data


networks:
  rung_internal:
    name: rung_internal
    driver: bridge
```

### Разбор ключевых решений

**Почему `$${POSTGRES_USER}` (с двумя долларами):** Docker Compose сам подставляет переменные `${VAR}` из `.env`. Чтобы переменная подставилась **внутри** контейнера (в момент выполнения команды), а не на этапе парсинга YAML, нужно экранировать доллар — `$$`. Без этого `pg_isready -U ${POSTGRES_USER}` попытается выполниться с буквальной строкой `${POSTGRES_USER}` и упадёт.

**Почему `condition: service_healthy`:** `app` не запустится, пока `db` не пройдёт healthcheck. Иначе приложение стартует раньше БД и падает с `connection refused`. Это самая частая ошибка у новичков.

**Почему `healthcheck` для Postgres — `pg_isready`:** это стандартная утилита Postgres. Отвечает за секунды, проверяет, что БД готова принимать соединения (не просто порт открыт).

**Почему `volumes: ../backend:/app/backend`:** код монтируется из хоста. Меняете файл в VS Code — контейнер видит изменения. Без этого пришлось бы пересобирать образ после каждой правки.

**Почему `restart: unless-stopped`:** если контейнер упадёт — Docker перезапустит его. Но если вы сами остановили (`docker compose stop`) — не перезапустит.

**Почему `depends_on` только от `db`:** сейчас других сервисов нет. Redis, Qdrant, Ollama появятся на этапах 7, 11 — тогда добавим их в `depends_on`.

**Почему `build.context: ..`:** контекст сборки — корень проекта (там, где `pyproject.toml`). Dockerfile лежит в `infrastructure/`, но ему нужны файлы из корня.

### Обновление `.env` для Docker

**Ключевой момент:** внутри контейнеров `localhost` — это сам контейнер, а не хост-машина и не другой контейнер. Поэтому адреса меняются.

Создайте `infrastructure/.env.example`:

```dotenv
# Application
DEBUG=true
PROJECT_NAME=RUNG
EVAL_MODE=full
ACTIVE_LEARNING_ENABLED=true

# Translation engine
TRANSLATION_ENGINE=auto
FALLBACK_TO_HF_ON_ERROR=true

# Ollama (пока не используется, но добавим для будущих этапов)
OLLAMA_HOST=ollama
OLLAMA_PORT=11434
LLM_MODEL=qwen2.5:7b
CRITIC_MODEL=qwen2.5:7b
EDITOR_MODEL=qwen2.5:7b
VALIDATOR_MODEL=qwen2.5:7b
ENSEMBLE_MODELS=qwen2.5:7b,llama3.1:8b

# Hugging Face
HF_TOKEN=your_huggingface_token_here
HF_TRANSLATION_MODEL=facebook/nllb-200-distilled-600M
HF_CRITIC_MODEL=Qwen/Qwen2.5-7B-Instruct
HF_EDITOR_MODEL=Qwen/Qwen2.5-7B-Instruct
HF_VALIDATOR_MODEL=Qwen/Qwen2.5-7B-Instruct

# Qdrant (пока не используется)
QDRANT_HOST=qdrant
QDRANT_PORT=6333
QDRANT_COLLECTION_NAME=tm_vectors

# Redis (пока не используется)
REDIS_URL=redis://redis:6379/0

# Database (ВНУТРИ DOCKER ХОСТ = имя сервиса, не localhost!)
POSTGRES_USER=rung_user
POSTGRES_PASSWORD=rung_password_123
POSTGRES_DB=rung_db
DATABASE_URL=postgresql://rung_user:rung_password_123@db:5432/rung_db

# Security (сгенерируйте свой ключ!)
JWT_SECRET_KEY=change_me_to_a_long_random_string
JWT_ALGORITHM=HS256
JWT_EXPIRATION_MINUTES=10080
```

Создайте рабочий `infrastructure/.env`:

```powershell
Copy-Item infrastructure\.env.example infrastructure\.env
```

Откройте `infrastructure/.env` и **замените** `JWT_SECRET_KEY` и `HF_TOKEN` на свои.

**Ключевое отличие от локального `.env`:**
- локально: `DATABASE_URL=sqlite+aiosqlite:///./rung.db`
- в Docker: `DATABASE_URL=postgresql://rung_user:...@db:5432/rung_db`

Хост `db` — это имя сервиса в compose. Docker сам создаёт DNS-запись `db` → IP контейнера.

### Создание `infrastructure/.dockerignore`

Файл `.dockerignore` не даёт копировать в образ лишние файлы (кэши, .venv, секреты).

Создайте `infrastructure/.dockerignore`:

```
# Git
.git
.gitignore

# Python
__pycache__/
*.py[cod]
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Virtual environments
.venv/
venv/

# IDE
.vscode/
.idea/

# Secrets
.env
.env.*
!.env.example

# Logs
logs/
*.log

# Local data
data/
*.db
*.sqlite
*.sqlite3

# ML caches / models
.cache/
.huggingface/
*.pt
*.pth
*.bin
*.safetensors

# OS
.DS_Store
Thumbs.db
```

### Коммит

```powershell
git add infrastructure/
git commit -m "feat(week-3): add Dockerfile and docker-compose (db + app)"
```

---

## 3.7. Запуск Docker Compose

**Проверьте, что Docker Desktop запущен.** Иконка в трее должна быть активна (зелёная).

### Запуск

Перейдите в папку `infrastructure`:

```powershell
cd infrastructure
```

**Запустите compose:**

```powershell
docker compose up -d --build
```

**Что вы должны увидеть:**

```
[+] Building ...
[+] Running 3/3
 ✔ Network rung_internal          Created
 ✔ Container infrastructure-db-1  Healthy
 ✔ Container infrastructure-app-1 Started
```

**Первый запуск займёт 3–5 минут** — скачивается образ Postgres (~80 МБ), собирается образ приложения, устанавливаются зависимости. Последующие запуски — секунды.

### Проверка статуса

```powershell
docker compose ps
```

**Что вы должны увидеть:**

```
NAME                       STATUS                    PORTS
infrastructure-app-1       Up (healthy)              0.0.0.0:8000->8000/tcp
infrastructure-db-1        Up (healthy)              5432/tcp
```

Оба сервиса `healthy` — критерий готовности выполнен.

### Просмотр логов

```powershell
docker compose logs app
```

**Что вы должны увидеть:**

```
app-1  | INFO:     Uvicorn running on http://0.0.0.0:8000
app-1  | INFO:     Application startup complete.
app-1  | 2026-09-11 16:00:00 - backend.api.main - INFO - Creating database tables...
app-1  | 2026-09-11 16:00:01 - backend.api.main - INFO - Database tables ready
```

**Если что-то не так:**

```powershell
docker compose logs db
```

Покажет логи Postgres — обычно там ошибки типа «не удалось создать директорию» или «пароль не тот».

### Проверка `/health`

```powershell
curl http://localhost:8000/health
```

**Что вы должны увидеть:**

```json
{"status":"healthy","project":"RUNG","model":"qwen2.5:7b","debug":true,"services":{"database":{"healthy":true,"error":null}}}
```

`database: healthy` — значит, приложение подключилось к Postgres в контейнере.

### Проверка БД напрямую

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "\dt"
```

**Что вы должны увидеть:**

```
             List of relations
 Schema |       Name           | Type  |   Owner
--------+----------------------+-------+-----------
 public | glossaries           | table | rung_user
 public | translation_cache    | table | rung_user
 public | translation_memories | table | rung_user
 public | uncertain_examples   | table | rung_user
 public | users                | table | rung_user
```

Пять таблиц — все на месте.

### Проверка сохранения данных (volume)

**Вставьте тестовую строку:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO users (id, email, username, hashed_password, role, is_active) VALUES ('test-1', 'a@b.c', 'testuser', 'x', 'user', true);"
```

**Проверьте, что строка есть:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, email FROM users;"
```

**Перезапустите контейнер `db`:**

```powershell
docker compose restart db
```

**Подождите 10 секунд и проверьте снова:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, email FROM users;"
```

**Что вы должны увидеть:** та же строка `test-1`. Значит volume работает — данные не потерялись при перезапуске.

**Удалите тестовую строку:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM users WHERE id = 'test-1';"
```

### Остановка

```powershell
docker compose down
```

**Важно:** `down` **не удаляет** volume. Данные сохраняются. Если хотите удалить всё (включая данные): `docker compose down -v` — осторожно, все данные пропадут.

### Коммит

Ничего нового не создано, но зафиксируйте состояние:

```powershell
git status
```

Скорее всего, чисто (все файлы закоммичены ранее).

---

## 3.8. Тесты — `tests/unit/test_models.py`

**Что тестируем:** модели SQLAlchemy создаются, вставляются, читаются, ограничения работают.

**Почему SQLite in-memory:** тесты должны работать быстро и без Docker. SQLite in-memory создаётся мгновенно, удаляется вместе с процессом, не оставляет следов.

Создайте `tests/unit/test_models.py`:

```python
# tests/unit/test_models.py
"""
Юнит-тесты для backend.db.models.

Используется SQLite in-memory — быстро, без Docker, без Postgres.
"""

import pytest
from sqlalchemy import create_engine, select
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import sessionmaker

from backend.db.models import (
    Base,
    GlossaryDB,
    TMDB,
    TranslationCache,
    UncertainExample,
    User,
)


@pytest.fixture
def session():
    """Создаёт in-memory SQLite и таблицы для каждого теста."""
    engine = create_engine("sqlite:///:memory:")
    Base.metadata.create_all(bind=engine)
    Session = sessionmaker(bind=engine)
    s = Session()
    try:
        yield s
    finally:
        s.close()
        engine.dispose()


# ============================================================================
# User
# ============================================================================
class TestUser:
    def test_create_user(self, session):
        u = User(
            id="u1",
            email="a@b.c",
            username="alice",
            hashed_password="hash",
            role="user",
            is_active=True,
        )
        session.add(u)
        session.commit()

        loaded = session.execute(select(User).where(User.id == "u1")).scalar_one()
        assert loaded.email == "a@b.c"
        assert loaded.username == "alice"
        assert loaded.role == "user"

    def test_email_unique(self, session):
        """Два пользователя с одинаковым email — IntegrityError."""
        session.add(User(id="u1", email="a@b.c", username="a", hashed_password="h"))
        session.commit()

        session.add(User(id="u2", email="a@b.c", username="b", hashed_password="h"))
        with pytest.raises(IntegrityError):
            session.commit()

    def test_username_unique(self, session):
        """Два пользователя с одинаковым username — IntegrityError."""
        session.add(User(id="u1", email="a@b.c", username="alice", hashed_password="h"))
        session.commit()

        session.add(User(id="u2", email="b@c.d", username="alice", hashed_password="h"))
        with pytest.raises(IntegrityError):
            session.commit()

    def test_role_default(self, session):
        """role по умолчанию = 'user'."""
        u = User(
            id="u1",
            email="a@b.c",
            username="alice",
            hashed_password="hash",
        )
        session.add(u)
        session.commit()
        assert u.role == "user"


# ============================================================================
# TMDB
# ============================================================================
class TestTMDB:
    def test_create_entry(self, session):
        entry = TMDB(
            id="en|ru|hello|привет",
            source_lang="en",
            target_lang="ru",
            source_text="hello",
            target_text="привет",
        )
        session.add(entry)
        session.commit()

        loaded = session.execute(
            select(TMDB).where(TMDB.source_text == "hello")
        ).scalar_one()
        assert loaded.target_text == "привет"

    def test_unique_constraint(self, session):
        """Повторная вставка той же пары должна упасть."""
        session.add(TMDB(
            id="id1",
            source_lang="en",
            target_lang="ru",
            source_text="hello",
            target_text="привет",
        ))
        session.commit()

        session.add(TMDB(
            id="id2",
            source_lang="en",
            target_lang="ru",
            source_text="hello",
            target_text="привет",
        ))
        with pytest.raises(IntegrityError):
            session.commit()

    def test_different_pairs_ok(self, session):
        """Разные языковые пары с одинаковым текстом — не конфликт."""
        session.add(TMDB(
            id="id1",
            source_lang="en",
            target_lang="ru",
            source_text="hello",
            target_text="привет",
        ))
        session.add(TMDB(
            id="id2",
            source_lang="en",
            target_lang="tt",
            source_text="hello",
            target_text="салам",
        ))
        session.commit()

        rows = session.execute(select(TMDB)).scalars().all()
        assert len(rows) == 2


# ============================================================================
# GlossaryDB
# ============================================================================
class TestGlossaryDB:
    def test_create_entry(self, session):
        g = GlossaryDB(
            id="en|ru|headache|головная боль",
            source_lang="en",
            target_lang="ru",
            source_text="headache",
            target_text="головная боль",
        )
        session.add(g)
        session.commit()

        loaded = session.execute(
            select(GlossaryDB).where(GlossaryDB.source_text == "headache")
        ).scalar_one()
        assert loaded.target_text == "головная боль"


# ============================================================================
# UncertainExample
# ============================================================================
class TestUncertainExample:
    def test_create_with_defaults(self, session):
        ex = UncertainExample(
            id="ex1",
            source_text="hello",
            source_lang="en",
            target_lang="ru",
            machine_translation="привет",
            critic_score=5.0,
            validation_passed=False,
        )
        session.add(ex)
        session.commit()

        loaded = session.execute(
            select(UncertainExample).where(UncertainExample.id == "ex1")
        ).scalar_one()
        assert loaded.status == "pending"
        assert loaded.critic_score == 5.0
        assert loaded.validation_passed is False


# ============================================================================
# TranslationCache
# ============================================================================
class TestTranslationCache:
    def test_two_engines_same_text(self, session):
        """
        Один текст может иметь два кэша — для ollama и huggingface.
        UniqueConstraint включает engine_used.
        """
        session.add(TranslationCache(
            id="c1",
            source_text="hello",
            source_lang="en",
            target_lang="ru",
            translation="привет",
            model_used="qwen2.5:7b",
            engine_used="ollama",
        ))
        session.add(TranslationCache(
            id="c2",
            source_text="hello",
            source_lang="en",
            target_lang="ru",
            translation="Привет!",
            model_used="nllb-200",
            engine_used="huggingface",
        ))
        session.commit()

        rows = session.execute(select(TranslationCache)).scalars().all()
        assert len(rows) == 2

    def test_same_engine_conflict(self, session):
        """Тот же текст + тот же движок — конфликт."""
        session.add(TranslationCache(
            id="c1",
            source_text="hello",
            source_lang="en",
            target_lang="ru",
            translation="привет",
            model_used="qwen2.5:7b",
            engine_used="ollama",
        ))
        session.commit()

        session.add(TranslationCache(
            id="c2",
            source_text="hello",
            source_lang="en",
            target_lang="ru",
            translation="привет",
            model_used="qwen2.5:7b",
            engine_used="ollama",
        ))
        with pytest.raises(IntegrityError):
            session.commit()

    def test_hits_default(self, session):
        c = TranslationCache(
            id="c1",
            source_text="x",
            source_lang="en",
            target_lang="ru",
            translation="y",
            model_used="m",
            engine_used="ollama",
        )
        session.add(c)
        session.commit()
        assert c.hits == 1
```

### Разбор

**Почему `sqlite:///:memory:`:** быстрая in-memory БД, не пишет на диск, удаляется вместе с процессом. Идеально для тестов.

**Почему фикстура создаёт engine заново для каждого теста:** изоляция. Тест `test_email_unique` не влияет на `test_create_user`. Каждый тест — чистая БД.

**Почему `IntegrityError`:** SQLAlchemy оборачивает все ошибки целостности (UNIQUE, NOT NULL, FK) в это исключение. Ловим его и проверяем, что БД действительно отклонила вставку.

**Почему `scalar_one()`:** возвращает ровно одну строку или падает. Если строк 0 или >1 — тест упадёт с понятной ошибкой. Лучше, чем `first()`.

### Запуск

```powershell
python -m poetry run pytest tests/unit/test_models.py -v
```

**Что вы должны увидеть:**

```
tests/unit/test_models.py::TestUser::test_create_user PASSED
tests/unit/test_models.py::TestUser::test_email_unique PASSED
...
============================= 11 passed in 0.6s ==============================
```

### Коммит

```powershell
git add tests/unit/test_models.py
git commit -m "test(week-3): add model tests with in-memory SQLite"
```

---

## 3.9. Финальная проверка этапа

### 1. Тесты проходят

```powershell
python -m poetry run pytest tests/ -v
```

**Что вы должны увидеть:** все тесты (config + health + models) `PASSED`.

### 2. Локальный запуск работает

```powershell
python -m poetry run python run.py
```

Откройте `http://localhost:8000/health` → `database: healthy`. Остановите (**Ctrl+C**).

### 3. Docker запускается

```powershell
cd infrastructure
docker compose up -d --build
docker compose ps
```

Оба сервиса `healthy`.

### 4. БД в контейнере доступна

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "\dt"
```

Пять таблиц.

### 5. `/health` в Docker работает

```powershell
curl http://localhost:8000/health
```

`database: healthy`.

### 6. Данные сохраняются

Повторите тест из 3.7 (вставка, рестарт, проверка).

### 7. Остановка

```powershell
docker compose down
cd ..
```

### 8. Слияние в main

```powershell
git add .
git commit -m "chore(week-3): finalize week 3"   # если есть что коммитить
git checkout main
git merge week-3 --no-ff -m "merge: week-3 (database + minimal Docker)"
git push origin main
git branch -d week-3
```

---

## 3.10. CRUD и другие команды для работы с БД через `docker compose exec`

На этом этапе **API для работы с данными ещё нет** — админ-роуты появятся только на этапе 9. Значит, единственный способ проверить, что таблицы действительно работают, вставить тестовые данные или очистить что-то по ошибке — это `psql` в контейнере `db`.

**Все команды ниже выполняются из папки `infrastructure`** (там, где `docker-compose.yml`). Если вы в корне проекта — используйте `docker compose -f infrastructure/docker-compose.yml exec ...`.

**Общий шаблон:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SQL-ЗАПРОС"
```

### CREATE — вставка данных

**Создать пользователя:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO users (id, email, username, hashed_password, full_name, role, is_active) VALUES ('u-001', 'admin@example.com', 'admin', 'fake_hash', 'Admin User', 'admin', true);"
```

**Добавить запись в TM (translation memory):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO translation_memories (id, source_lang, target_lang, source_text, target_text) VALUES ('en|ru|hello|привет', 'en', 'ru', 'hello', 'привет');"
```

**Добавить термин в глоссарий:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO glossaries (id, source_lang, target_lang, source_text, target_text) VALUES ('en|ru|headache|головная боль', 'en', 'ru', 'headache', 'головная боль');"
```

**Добавить неуверенный перевод в очередь HITL:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO uncertain_examples (id, source_text, source_lang, target_lang, machine_translation, critic_score, validation_passed, status) VALUES ('ex-001', 'The patient has a headache.', 'en', 'ru', 'Пациент имеет головную боль.', 5.0, false, 'pending');"
```

**Добавить запись в кэш:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO translation_cache (id, source_text, source_lang, target_lang, translation, model_used, engine_used, hits) VALUES ('c-001', 'hello', 'en', 'ru', 'привет', 'qwen2.5:7b', 'ollama', 1);"
```

**Вставить несколько записей одной командой:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO translation_memories (id, source_lang, target_lang, source_text, target_text) VALUES ('en|ru|good morning|доброе утро', 'en', 'ru', 'good morning', 'доброе утро'), ('en|ru|good night|доброй ночи', 'en', 'ru', 'good night', 'доброй ночи'), ('en|ru|thank you|спасибо', 'en', 'ru', 'thank you', 'спасибо');"
```

### READ — чтение данных

**Все пользователи:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, email, username, role, is_active FROM users;"
```

**Первые 20 записей TM:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, source_lang, target_lang, source_text, target_text FROM translation_memories LIMIT 20;"
```

**TM конкретной языковой пары:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_text, target_text FROM translation_memories WHERE source_lang = 'en' AND target_lang = 'ru' LIMIT 50;"
```

**Глоссарий конкретной пары:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_text, target_text FROM glossaries WHERE source_lang = 'en' AND target_lang = 'ru';"
```

**Очередь HITL (только pending):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, source_text, critic_score, status FROM uncertain_examples WHERE status = 'pending' ORDER BY created_at DESC LIMIT 20;"
```

**Кэш с наибольшим числом попаданий:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_text, translation, engine_used, hits FROM translation_cache ORDER BY hits DESC LIMIT 10;"
```

**Количество записей по каждой таблице:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT 'users' AS tbl, COUNT(*) FROM users UNION ALL SELECT 'translation_memories', COUNT(*) FROM translation_memories UNION ALL SELECT 'glossaries', COUNT(*) FROM glossaries UNION ALL SELECT 'translation_cache', COUNT(*) FROM translation_cache UNION ALL SELECT 'uncertain_examples', COUNT(*) FROM uncertain_examples;"
```

**Количество записей TM по языковым парам:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_lang, target_lang, COUNT(*) FROM translation_memories GROUP BY source_lang, target_lang ORDER BY source_lang, target_lang;"
```

**Количество записей глоссария по языковым парам:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_lang, target_lang, COUNT(*) FROM glossaries GROUP BY source_lang, target_lang ORDER BY source_lang, target_lang;"
```

**Статистика кэша (записей и попаданий по движкам):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT engine_used, COUNT(*) AS entries, SUM(hits) AS total_hits FROM translation_cache GROUP BY engine_used;"
```

**Статистика HITL по статусам:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT status, COUNT(*) FROM uncertain_examples GROUP BY status;"
```

### UPDATE — изменение данных

**Сменить роль пользователя:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET role = 'editor' WHERE email = 'admin@example.com';"
```

**Деактивировать пользователя:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET is_active = false WHERE email = 'spam@example.com';"
```

**Исправить перевод в TM:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE translation_memories SET target_text = 'здравствуйте' WHERE source_lang = 'en' AND target_lang = 'ru' AND source_text = 'hello';"
```

**Подтвердить исправление в HITL (статус → confirmed):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE uncertain_examples SET status = 'confirmed', corrected_translation = 'У пациента головная боль.', confirmed_at = NOW() WHERE id = 'ex-001';"
```

**Отклонить пример в HITL:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE uncertain_examples SET status = 'rejected' WHERE id = 'ex-001';"
```

**Сбросить счётчик попаданий в кэше:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE translation_cache SET hits = 0;"
```

### DELETE — удаление данных

**Удалить одного пользователя по email:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM users WHERE email = 'spam@example.com';"
```

**Удалить конкретную запись TM по ID:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM translation_memories WHERE id = 'en|ru|hello|привет';"
```

**Удалить всю языковую пару из TM:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM translation_memories WHERE source_lang = 'en' AND target_lang = 'tt';"
```

**Удалить несколько языковых пар из TM одной командой:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM translation_memories WHERE (source_lang = 'en' AND target_lang = 'tt') OR (source_lang = 'ru' AND target_lang = 'tt');"
```

**Удалить пару из глоссария:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM glossaries WHERE (source_lang = 'en' AND target_lang = 'tt') OR (source_lang = 'ru' AND target_lang = 'tt');"
```

**Удалить пару из кэша:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM translation_cache WHERE (source_lang = 'en' AND target_lang = 'tt') OR (source_lang = 'ru' AND target_lang = 'tt');"
```

**Удалить пару из очереди HITL:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM uncertain_examples WHERE (source_lang = 'en' AND target_lang = 'tt') OR (source_lang = 'ru' AND target_lang = 'tt');"
```

**Очистить таблицу целиком (TRUNCATE — быстрее, чем DELETE):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "TRUNCATE TABLE translation_memories;"
```

**Очистить кэш:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "TRUNCATE TABLE translation_cache;"
```

**Очистить всю очередь HITL:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "TRUNCATE TABLE uncertain_examples;"
```

### Комбинированные сценарии

**Полная очистка языковой пары `en→tt` из всех таблиц одной командой:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM translation_memories WHERE source_lang = 'en' AND target_lang = 'tt'; DELETE FROM glossaries WHERE source_lang = 'en' AND target_lang = 'tt'; DELETE FROM translation_cache WHERE source_lang = 'en' AND target_lang = 'tt'; DELETE FROM uncertain_examples WHERE source_lang = 'en' AND target_lang = 'tt';"
```

`psql -c` принимает несколько команд через `;`. Каждая выведет `DELETE N` — сколько строк удалено.

**Посмотреть содержимое всех таблиц по очереди:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT 'users' AS tbl, COUNT(*) FROM users; SELECT 'translation_memories', COUNT(*) FROM translation_memories; SELECT 'glossaries', COUNT(*) FROM glossaries; SELECT 'translation_cache', COUNT(*) FROM translation_cache; SELECT 'uncertain_examples', COUNT(*) FROM uncertain_examples;"
```

**Полная перезагрузка БД (удалить все данные, сохранить схему):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "TRUNCATE TABLE users, translation_memories, glossaries, translation_cache, uncertain_examples CASCADE;"
```

### Резервное копирование и восстановление

**Сохранить всю БД в файл:**

```powershell
docker compose exec db pg_dump -U rung_user -d rung_db > backup.sql
```

Файл `backup.sql` появится в текущей папке (`infrastructure/`). Добавьте его в `.gitignore`, если не хотите коммитить.

**Восстановить из файла:**

```powershell
Get-Content backup.sql | docker compose exec -T db psql -U rung_user -d rung_db
```

Флаг `-T` обязателен — без него `psql` будет требовать TTY и упадёт в пайпе.

### Полезные `psql`-метакоманды

**Список таблиц:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "\dt"
```

**Структура конкретной таблицы:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "\d translation_memories"
```

**Размер всех таблиц:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT relname AS table, pg_size_pretty(pg_total_relation_size(relid)) AS size FROM pg_catalog.pg_statio_user_tables ORDER BY pg_total_relation_size(relid) DESC;"
```

**Список индексов:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "\di"
```

**Активные соединения:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT count(*) FROM pg_stat_activity WHERE datname = 'rung_db';"
```

### Интерактивный режим

Если команд много и хочется «поработать как в консоли» — откройте интерактивную сессию:

```powershell
docker compose exec db psql -U rung_user -d rung_db
```

Внутри можно писать SQL-запросы без обёртки `-c "..."`, использовать стрелки вверх для истории, `\dt`, `\d table`, и так далее. Выход — `\q` или `Ctrl+D`.

**Полезно, когда нужно:**
- быстро посмотреть много таблиц подряд;
- выполнить 10 запросов;
- отладить один сложный SELECT.

### Зачем всё это на этапе 3

Три практических сценария:

**1. Проверить, что таблицы действительно работают.** Создали таблицу — вставили строку — прочитали. Если работает, значит, `create_all` и модели написаны правильно.

**2. Подготовить данные для следующих этапов.** На этапе 4 (Auth) понадобится пользователь — можно вставить его через `psql` до того, как появится `/auth/register`. На этапе 10 (HITL) понадобятся примеры в очереди — можно набить их вручную.

**3. Отладка.** Когда что-то не работает, первым делом смотрите данные: `SELECT * FROM translation_cache LIMIT 5;`. Часто причина находится сразу.

---

## 3.11. Структура проекта после Этапа 3

```
RUNG/
├── .editorconfig
├── .env                          # локально, не в Git
├── .env.example
├── .gitattributes
├── .gitignore
├── README.md
├── poetry.lock
├── pyproject.toml
├── run.py
├── rung.db                       # локальная SQLite, не в Git
│
├── backend/
│   ├── __init__.py
│   ├── api/
│   │   ├── __init__.py
│   │   ├── main.py                            ← обновлён (create_all в lifespan)
│   │   └── routes/
│   │       ├── __init__.py
│   │       └── health.py                      ← обновлён (/health проверяет БД)
│   ├── core/
│   │   ├── __init__.py
│   │   ├── config.py
│   │   ├── constants.py
│   │   └── exceptions.py
│   ├── db/
│   │   ├── __init__.py
│   │   ├── session.py                         ← новый (week-3)
│   │   ├── models.py                          ← новый (week-3)
│   │   └── database.py                        ← новый (week-3)
│   ├── utils/
│   │   ├── __init__.py
│   │   └── logging.py
│   ├── agents/__init__.py                     # пусто (этап 5)
│   ├── data/__init__.py                       # пусто (этап 6)
│   ├── hitl/__init__.py                       # пусто (этап 10)
│   ├── retrieval/__init__.py                  # пусто (этап 7)
│   └── worker/__init__.py                     # пусто (этап 11)
│
├── data/
│   └── import/
│       └── .gitkeep
│
├── infrastructure/
│   ├── Dockerfile                             ← новый (week-3)
│   ├── docker-compose.yml                     ← новый (week-3)
│   ├── .dockerignore                          ← новый (week-3)
│   ├── .env                                   # локально, не в Git
│   └── .env.example                           ← новый (week-3)
│
├── logs/
│   ├── .gitkeep
│   ├── rung_YYYYMMDD.log
│   └── errors_YYYYMMDD.log
│
├── scripts/                                   # пусто (этапы 6, 9)
│
└── tests/
    ├── __init__.py
    ├── integration/
    │   ├── __init__.py
    │   └── test_health.py
    └── unit/
        ├── __init__.py
        ├── test_config.py
        └── test_models.py                     ← новый (week-3)
```

### Как проверить

```powershell
tree /F /A
```

### Каких файлов ещё НЕ должно быть

- `backend/utils/security.py`, `backend/api/deps.py`, `backend/api/routes/auth.py` — этап 4.
- `backend/agents/*` (кроме `__init__.py`) — этап 5.
- `backend/data/*` (кроме `__init__.py`) — этапы 6, 7.
- `backend/retrieval/*` — этап 7.
- `backend/hitl/*` — этап 10.
- `backend/eval/*` — этап 9б.
- `backend/worker/celery_app.py`, `backend/worker/tasks.py` — этап 11.
- `infrastructure/nginx.conf` — этап 12.
- `streamlit_app/` — этап 12.

---

## 3.12. Troubleshooting

### `ModuleNotFoundError: No module named 'aiosqlite'` (или `asyncpg`)

**Причина:** зависимости не установлены.

**Решение:** `python -m poetry install`.

### `RuntimeError: Task got Future attached to a different loop`

**Симптом:** падает при первом запросе к БД, обычно в Celery или при нестандартном event loop.

**Причина:** SQLAlchemy async engine привязан к event loop, в котором создан. Если приложение запускает новый loop (например, в Celery), движок перестаёт работать.

**Решение:** на этом этапе проблема не должна появиться. Если появилась — убедитесь, что `async_engine` создаётся **один раз на процесс**, при импорте модуля. Не создавайте engine внутри функции.

### `asyncpg.exceptions.InvalidPasswordError` (или аналог)

**Симптом:** в Docker `app` не может подключиться к `db`.

**Причина:** в `DATABASE_URL` опечатка или пароль не совпадает с `POSTGRES_PASSWORD`.

**Решение:** проверьте `infrastructure/.env` — `POSTGRES_PASSWORD` и пароль в `DATABASE_URL` должны совпадать. Также хост должен быть `db` (имя сервиса), а не `localhost`.

### `connection refused` к `db:5432` в Docker

**Причина:** `app` стартует до того, как Postgres готов принимать соединения.

**Решение:** у `app` в compose должен быть `depends_on: db: condition: service_healthy`. Если этот блок есть, а ошибка остаётся — значит `healthcheck` не проходит. Проверьте логи `db`: `docker compose logs db`.

### `docker compose` не находит `docker-compose.yml`

**Причина:** вы не в папке `infrastructure`.

**Решение:** `cd infrastructure` или используйте флаг: `docker compose -f infrastructure/docker-compose.yml up`.

### `bind: address already in use` на порту 8000 или 5432

**Причина:** порт занят другим процессом (другой контейнер, локальный uvicorn, локальный Postgres).

**Решение:**

```powershell
netstat -ano | findstr :8000
taskkill /PID <PID> /F
```

Или смените порт в `docker-compose.yml`: `8001:8000`.

### `permission denied` на папке `logs`

**Симптом:** контейнер не может писать в `logs/`.

**Решение:** на Windows с Docker Desktop это редкость, но если возникло — создайте папку с открытыми правами: `mkdir logs` в корне проекта (если её нет).

### `Cannot connect to the Docker daemon`

**Причина:** Docker Desktop не запущен.

**Решение:** запустите Docker Desktop, дождитесь, пока иконка в трее станет зелёной.

### Тесты падают с `IntegrityError` там, где не должны

**Симптом:** `test_email_unique` не падает, потому что SQLite по умолчанию **не** проверяет UNIQUE-ограничения... хотя должен.

**Решение:** SQLite in-memory всё-таки проверяет UNIQUE. Если ваш тест не падает — проверьте, что вы действительно вставляете дубликат (одинаковый `email`). Если и это не помогает — это баг в SQLite-драйвере (маловероятно).

### `sqlalchemy.exc.MissingGreenlet`

**Симптом:** атрибут объекта пустой после `commit()`.

**Причина:** `expire_on_commit=True` (по умолчанию). После коммита объект «протухает» и пытается перезагрузиться из БД — а это уже async-операция.

**Решение:** у нас `expire_on_commit=False` в `async_sessionmaker`. Если ошибка всё равно появляется — проверьте, что сессия создаётся именно через `AsyncSessionLocal`, а не через другой sessionmaker.

---

## Итог этапа 3

К концу этапа у вас есть:

**Файлы кода:**

- `backend/db/session.py` — async engine, session factory, FastAPI dependency;
- `backend/db/models.py` — 5 моделей SQLAlchemy;
- `backend/db/database.py` — реэкспорт;
- `backend/api/main.py` — обновлённый `lifespan` с `create_all`;
- `backend/api/routes/health.py` — `/health` с проверкой БД;
- `infrastructure/Dockerfile` — минимальный образ;
- `infrastructure/docker-compose.yml` — `db` + `app`;
- `infrastructure/.env.example` и `infrastructure/.env`;
- `infrastructure/.dockerignore`.

**Тесты:**

- `tests/unit/test_models.py` — 11 тестов на модели с SQLite in-memory.

**Что работает:**

- `python -m poetry run python run.py` — приложение стартует, создаёт таблицы в SQLite.
- `docker compose -f infrastructure/docker-compose.yml up -d --build` — поднимает `db` и `app`, оба `healthy`.
- `/health` возвращает `database: healthy`.
- Данные в Postgres сохраняются при перезапуске (volume).
- 11 новых тестов + старые 25 проходят.
- Все CRUD-операции доступны через `psql` в контейнере `db`.

**Что дальше — Этап 4: Аутентификация и авторизация**

На следующем этапе вы:

- реализуете `backend/utils/security.py` — bcrypt + JWT;
- создадите `backend/api/deps.py` — `get_current_user`, `require_admin`, `require_editor`;
- напишете `backend/api/routes/auth.py` — `/auth/register`, `/auth/login`, `/auth/me`;
- добавите Pydantic-модели `RegisterRequest`, `LoginRequest`, `TokenResponse`;
- напишете тесты на регистрацию, логин, роли.